# Internet Rabbit Hole Dataset
## Building a Wikipedia Navigation Dataset from Real Human Click Data

**Source:** [Wikimedia Clickstream](https://dumps.wikimedia.org/other/clickstream/) — English Wikipedia, Jan–May 2026  
**License:** CC BY-SA 4.0  
**GitHub:** https://github.com/vansh-kumar-007/Internet-Rabbit-Hole-Dataset

### What this notebook does
1. Downloads the Wikimedia clickstream dump (real human Wikipedia clicks)
2. Builds a memory-efficient click-weighted graph (CSR format, ~220 MB RAM)
3. Reconstructs 50,000 rabbit hole paths via weighted random walks
4. Engineers three topic drift features using TF-IDF cosine distance
5. Saves the final dataset as Parquet

### Example record
```json
{
  "session_id": "a7f3c...",
  "start_article": "Aerodynamics",
  "end_article": "Roman_Empire",
  "path": ["Aerodynamics", "Wright_brothers", "..."],
  "path_length": 12,
  "drift_score": 0.91,
  "max_step_drift": 1.0,
  "semantic_spread": 0.74
}
```

## 1. Environment Verification

Confirm Python version and all required libraries before starting.

In [ ]:
import sys, importlib

REQUIRED = [
    ('requests','requests'), ('bs4','beautifulsoup4'),
    ('pandas','pandas'), ('numpy','numpy'),
    ('tqdm','tqdm'), ('pyarrow','pyarrow'),
    ('sklearn','scikit-learn'),
]

print(f'Python {sys.version}\n')
all_ok = True
for imp, pip in REQUIRED:
    spec = importlib.util.find_spec(imp)
    if spec:
        mod = importlib.import_module(imp)
        print(f'  ✓  {pip:<20} {getattr(mod, "__version__", "ok")}')
    else:
        print(f'  ✗  {pip:<20} MISSING — run: pip install {pip}')
        all_ok = False
print('\n✓ Ready.' if all_ok else '\n✗ Install missing libraries first.')

## 2. Data Download

Download the Wikimedia English Wikipedia clickstream dump for May 2026.
Uses chunked streaming to keep memory flat during download.

In [ ]:
import requests, os
from tqdm import tqdm

DATA_DIR     = '/content/data'
TARGET_MONTH = '2026-05'
FILENAME     = f'clickstream-enwiki-{TARGET_MONTH}.tsv.gz'
DOWNLOAD_URL = (f'https://dumps.wikimedia.org/other/clickstream/'
                f'{TARGET_MONTH}/{FILENAME}')
FILE_PATH    = os.path.join(DATA_DIR, FILENAME)

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(FILE_PATH):
    size_mb = os.path.getsize(FILE_PATH) / (1024*1024)
    print(f'✓ Already on disk: {FILENAME} ({size_mb:.1f} MB)')
else:
    print(f'Downloading {FILENAME}...')
    r     = requests.get(DOWNLOAD_URL, stream=True, timeout=60)
    total = int(r.headers.get('Content-Length', 0))
    with open(FILE_PATH, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, ncols=70) as bar:
        for chunk in r.iter_content(1024*1024):
            f.write(chunk); bar.update(len(chunk))
    actual = os.path.getsize(FILE_PATH)
    assert actual == total, 'Size mismatch — file corrupt'
    print(f'✓ Downloaded {actual/(1024*1024):.1f} MB — integrity confirmed')

## 3. Stream Filter

Read the compressed file line-by-line and extract only `type=link` rows
(Wikipedia-to-Wikipedia clicks). This keeps memory usage flat regardless
of file size, and sanitizes embedded tab characters in article titles.

In [ ]:
import gzip, os
from tqdm import tqdm

FILTERED_TSV = os.path.join(DATA_DIR, f'links-only-enwiki-{TARGET_MONTH}.tsv')

if os.path.exists(FILTERED_TSV):
    rows = sum(1 for _ in open(FILTERED_TSV)) - 1  # subtract header
    print(f'✓ Already filtered: {rows:,} link rows on disk')
else:
    print('Streaming and filtering...')
    kept = bad = 0
    with gzip.open(FILE_PATH, 'rt', encoding='utf-8', errors='replace') as inf, \
         open(FILTERED_TSV, 'w', encoding='utf-8') as outf:
        outf.write('prev\tcurr\ttype\tn\n')
        for line in tqdm(inf, unit=' lines', ncols=70, mininterval=3.0):
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 4: bad += 1; continue
            n, link_type = parts[-1], parts[-2]
            curr = ' '.join(parts[1:-2])
            prev = parts[0]
            if link_type != 'link': continue
            if not n.strip().isdigit(): bad += 1; continue
            outf.write(
                f"{prev.replace(chr(9),' ').strip()}\t"
                f"{curr.replace(chr(9),' ').strip()}\t"
                f"link\t{n}\n"
            )
            kept += 1
    print(f'✓ Kept {kept:,} link rows  |  bad/skipped {bad:,}')

# Load into pandas with bulletproof all-string dtype
import pandas as pd
print('Loading into pandas...')
df = pd.read_csv(FILTERED_TSV, sep='\t', dtype=str,
                 on_bad_lines='skip', engine='c')
df.dropna(subset=['prev','curr','n'], inplace=True)
df = df[df['n'].str.strip().str.isdigit()].copy()
df['n'] = df['n'].str.strip().astype('int32')
df['type'] = df['type'].astype('category')
df.reset_index(drop=True, inplace=True)
print(f'✓ {len(df):,} clean edges  |  '
      f'RAM: {df.memory_usage(deep=True).sum()/1e6:.0f} MB')

# Save as Parquet and free TSV from disk
EDGE_PARQUET = os.path.join(DATA_DIR,
               f'clickstream-enwiki-{TARGET_MONTH}.parquet')
df.to_parquet(EDGE_PARQUET, engine='pyarrow',
              compression='snappy', index=False)
os.remove(FILTERED_TSV)
print(f'✓ Edge Parquet saved: '
      f'{os.path.getsize(EDGE_PARQUET)/1e6:.0f} MB  |  TSV deleted')

## 4. Memory-Efficient Graph Construction

Convert the edge list into a CSR (Compressed Sparse Row) graph using
numpy integer arrays. This represents 3.3M articles and 22M edges
in ~220 MB RAM — versus 4-6 GB with a Python dictionary approach.

**Structure:**
- `id_to_article` — integer → article name
- `unique_prev` — sorted array of source node IDs
- `curr_ids` — target node IDs (sorted by source)
- `counts` — click counts parallel to curr_ids
- `edge_start/end` — slice offsets into curr_ids per source node

In [ ]:
import numpy as np, pandas as pd, os, gc

# Load edge Parquet (fast, typed, compressed)
EDGE_PARQUET = os.path.join(DATA_DIR,
               f'clickstream-enwiki-{TARGET_MONTH}.parquet')
df = pd.read_parquet(EDGE_PARQUET, engine='pyarrow')
print(f'✓ Loaded {len(df):,} edges')

# Build integer vocabulary
all_arts    = pd.concat([df['prev'], df['curr']]).unique()
article_to_id = {a: i for i, a in enumerate(all_arts)}
id_to_article = np.array(all_arts)
print(f'✓ Vocabulary: {len(id_to_article):,} unique articles')

# Encode edges as integer arrays
prev_ids = np.array([article_to_id[a] for a in df['prev']], dtype=np.int32)
curr_ids = np.array([article_to_id[a] for a in df['curr']], dtype=np.int32)
counts   = df['n'].to_numpy(dtype=np.int32)
del df; gc.collect()

# Sort by source node for CSR slicing
idx      = np.argsort(prev_ids, kind='stable')
prev_ids = prev_ids[idx]
curr_ids = curr_ids[idx]
counts   = counts[idx]

# Build offset arrays
unique_prev, edge_start = np.unique(prev_ids, return_index=True)
edge_end = np.append(edge_start[1:], len(prev_ids))
del prev_ids

total_mb = (unique_prev.nbytes + curr_ids.nbytes + counts.nbytes +
            edge_start.nbytes + edge_end.nbytes +
            id_to_article.nbytes) / 1e6
print(f'✓ CSR graph built')
print(f'  Source nodes : {len(unique_prev):,}')
print(f'  Total edges  : {len(curr_ids):,}')
print(f'  RAM usage    : {total_mb:.0f} MB')

## 5. Random Walk Path Reconstruction

Generate 50,000 rabbit hole sessions using click-weighted random walks.

**Algorithm:** At each hop, choose the next article probabilistically,
weighted by real click counts. High-traffic links are followed more often,
mirroring real human browsing behavior. Walks stop when:
- Maximum hops reached (20)
- Dead end (article has no outgoing clicks)
- All neighbors already visited (loop prevention)

**Key finding:** 34.9% of paths reach the 20-hop maximum, confirming
Wikipedia's extraordinary graph density.

In [ ]:
import numpy as np, uuid, json, os
from tqdm import tqdm

MIN_PATH_LENGTH = 4
MAX_PATH_LENGTH = 20
N_WALKS         = 50_000
RANDOM_SEED     = 2024
JSONL_FILE      = os.path.join(DATA_DIR,
                  f'rabbit_holes_{TARGET_MONTH}.jsonl')

rng = np.random.default_rng(RANDOM_SEED)

def random_walk(start_id, rng):
    """Click-weighted random walk on the Wikipedia graph."""
    path, visited = [start_id], {start_id}
    for _ in range(MAX_PATH_LENGTH - 1):
        cur = path[-1]
        ix  = np.searchsorted(unique_prev, cur)
        if ix >= len(unique_prev) or unique_prev[ix] != cur:
            break
        s, e  = edge_start[ix], edge_end[ix]
        nbrs  = curr_ids[s:e]
        wts   = counts[s:e].astype(np.float32)
        mask  = np.array([n not in visited for n in nbrs])
        if not mask.any(): break
        nbrs, wts = nbrs[mask], wts[mask]
        probs = wts / wts.sum()
        nxt   = int(rng.choice(nbrs, p=probs))
        path.append(nxt); visited.add(nxt)
    return path

if os.path.exists(JSONL_FILE):
    existing = sum(1 for _ in open(JSONL_FILE))
    print(f'✓ Already generated: {existing:,} sessions on disk')
else:
    print(f'Generating {N_WALKS:,} rabbit hole sessions...')
    written = attempts = 0
    with open(JSONL_FILE, 'w') as f, \
         tqdm(total=N_WALKS, ncols=70, unit=' paths') as pbar:
        while written < N_WALKS:
            attempts += 1
            sid   = int(rng.choice(unique_prev))
            pids  = random_walk(sid, rng)
            if len(pids) < MIN_PATH_LENGTH: continue
            names = [id_to_article[i] for i in pids]
            rec   = {
                'session_id'   : str(uuid.uuid4()),
                'month'        : TARGET_MONTH,
                'start_article': names[0],
                'end_article'  : names[-1],
                'path'         : names,
                'path_length'  : len(names),
            }
            f.write(json.dumps(rec) + '\n')
            written += 1; pbar.update(1)
    print(f'✓ {written:,} sessions written '
          f'({attempts:,} attempts, '
          f'{written/attempts*100:.1f}% success rate)')
    size_mb = os.path.getsize(JSONL_FILE) / 1e6
    print(f'✓ JSONL: {size_mb:.1f} MB → {JSONL_FILE}')

## 6. Feature Engineering — Topic Drift Scores

Compute three semantic drift features for each rabbit hole path using
TF-IDF vectors of Wikipedia article titles.

| Feature | Description |
|---------|-------------|
| `drift_score` | Mean cosine distance between consecutive hops (0–1) |
| `max_step_drift` | Largest single topic jump in the path (0–1) |
| `semantic_spread` | Mean distance of all articles from path centroid (0–1) |

**Why TF-IDF on titles?** Wikipedia article titles are concise, information-dense topic labels. TF-IDF over unigrams+bigrams captures lexical topic similarity efficiently without requiring GPU or paid APIs.

**Key finding:** `drift_score` and `path_length` are nearly uncorrelated (r=0.026) — short paths can drift just as wildly as long ones.

In [ ]:
import json, re, os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from tqdm import tqdm

FINAL_PARQUET = os.path.join(DATA_DIR,
               f'internet_rabbit_holes_{TARGET_MONTH}.parquet')

if os.path.exists(FINAL_PARQUET):
    df_final = pd.read_parquet(FINAL_PARQUET)
    print(f'✓ Already computed: {len(df_final):,} enriched sessions')
    print(df_final[['drift_score','max_step_drift',
                     'semantic_spread']].describe().round(3).to_string())
else:
    # Load sessions
    records = [json.loads(l) for l in open(JSONL_FILE)]
    print(f'✓ Loaded {len(records):,} sessions')

    # Build TF-IDF corpus from all article titles in paths
    unique_arts = sorted({a for r in records for a in r['path']})
    art_to_idx  = {a: i for i, a in enumerate(unique_arts)}

    def clean_title(t):
        return re.sub(r'\(.*?\)', '', t.replace('_', ' ')).strip()

    corpus = [clean_title(a) for a in unique_arts]
    print(f'✓ Corpus: {len(corpus):,} article titles')

    vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2,
                                 max_features=50_000, sublinear_tf=True)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    print(f'✓ TF-IDF matrix: {tfidf_matrix.shape}')

    # Compute drift features per path
    def path_features(path):
        idxs  = [art_to_idx[a] for a in path if a in art_to_idx]
        if len(idxs) < 2: return None
        steps = [cosine_distances(tfidf_matrix[i], tfidf_matrix[j])[0][0]
                 for i, j in zip(idxs[:-1], idxs[1:])]
        vecs  = np.asarray(tfidf_matrix[idxs].todense())
        ctr   = vecs.mean(axis=0, keepdims=True)
        ctr  /= (np.linalg.norm(ctr) + 1e-9)
        spread = [float(cosine_distances(
                      vecs[i:i+1]/(np.linalg.norm(vecs[i:i+1])+1e-9),
                      ctr)[0][0]) for i in range(len(idxs))]
        return {
            'drift_score'    : round(float(np.mean(steps)), 4),
            'max_step_drift' : round(float(np.max(steps)), 4),
            'semantic_spread': round(float(np.mean(spread)), 4),
        }

    enriched = []
    for rec in tqdm(records, ncols=70, unit=' recs'):
        feats = path_features(rec['path'])
        if feats: enriched.append({**rec, **feats})
    print(f'✓ Features computed: {len(enriched):,} records')

    # Save final Parquet
    df_final = pd.DataFrame(enriched)
    df_final['path_length']     = df_final['path_length'].astype('int16')
    df_final['drift_score']     = df_final['drift_score'].astype('float32')
    df_final['max_step_drift']  = df_final['max_step_drift'].astype('float32')
    df_final['semantic_spread'] = df_final['semantic_spread'].astype('float32')
    df_final['month']           = df_final['month'].astype('category')
    df_final.to_parquet(FINAL_PARQUET, engine='pyarrow',
                        compression='snappy', index=False)
    mb = os.path.getsize(FINAL_PARQUET) / 1e6
    print(f'✓ Saved {len(df_final):,} rows → {FINAL_PARQUET} ({mb:.1f} MB)')
    print(df_final[['drift_score','max_step_drift',
                     'semantic_spread']].describe().round(3).to_string())

## 7. Exploratory Data Analysis

Validate the final dataset and visualize key distributions.

### Dataset at a glance
- **50,000** rabbit hole sessions from real Wikipedia click data
- **665,921** total article visits across all paths
- **49,346** unique starting articles
- **34.9%** of paths reach the 20-hop maximum

### The 'Getting to Philosophy' phenomenon
The top rabbit hole attractors — `Philosophy`, `Linguistics`,
`Ancient_Greek`, `Abstraction` — emerge naturally from real click
data, independently confirming the well-known Wikipedia convergence
phenomenon.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
from collections import Counter

FINAL_PARQUET = os.path.join(DATA_DIR,
               f'internet_rabbit_holes_{TARGET_MONTH}.parquet')
df = pd.read_parquet(FINAL_PARQUET)
print(f'✓ Loaded {len(df):,} sessions\n')

# ── Summary statistics ────────────────────────────────
numeric = ['path_length','drift_score','max_step_drift','semantic_spread']
print('Feature statistics:')
print(df[numeric].describe().round(3).to_string())

print(f'\nKey facts:')
print(f'  Total article visits : {df["path"].apply(len).sum():,}')
print(f'  Unique start articles: {df["start_article"].nunique():,}')
print(f'  Unique end articles  : {df["end_article"].nunique():,}')
print(f'  Paths at max length  : '
      f'{(df["path_length"]==20).sum():,} '
      f'({(df["path_length"]==20).mean()*100:.1f}%)')

print('\nTop 10 rabbit hole attractors (end articles):')
for art, cnt in Counter(df['end_article']).most_common(10):
    print(f'  {cnt:>4}x  {art}')

# ── Visualizations ────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'#f8f8f8',
    'axes.grid':True, 'grid.color':'white', 'grid.linewidth':1.2,
    'axes.spines.top':False, 'axes.spines.right':False,
})
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Internet Rabbit Hole Dataset — May 2026\n'
    '50,000 Wikipedia Navigation Sessions',
    fontsize=14, fontweight='bold', y=1.02
)

# Plot 1: Path length distribution
ax1 = axes[0]
bins = np.arange(4, 22) - 0.5
cnts, _, patches = ax1.hist(df['path_length'], bins=bins,
                             color='#4C72B0', edgecolor='white')
patches[-1].set_facecolor('#C44E52')
ax1.set_xlabel('Path Length (hops)'); ax1.set_ylabel('Sessions')
ax1.set_title('Path Length Distribution', fontweight='bold')
ax1.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

# Plot 2: Drift score distribution
ax2 = axes[1]
scores = df['drift_score'].values
ax2.hist(scores, bins=50, color='#55A868', edgecolor='white')
ax2.axvline(scores.mean(), color='#C44E52', lw=2, ls='--',
            label=f'Mean={scores.mean():.3f}')
ax2.axvline(np.median(scores), color='#4C72B0', lw=2, ls=':',
            label=f'Median={np.median(scores):.3f}')
ax2.set_xlabel('Drift Score'); ax2.set_ylabel('Sessions')
ax2.set_title('Topic Drift Score Distribution', fontweight='bold')
ax2.legend(fontsize=9)

# Plot 3: Correlation heatmap
ax3 = axes[2]
corr   = df[numeric].corr()
labels = ['Path\nLength','Drift\nScore',
          'Max Step\nDrift','Semantic\nSpread']
im = ax3.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax3.set_xticks(range(4)); ax3.set_yticks(range(4))
ax3.set_xticklabels(labels, fontsize=9)
ax3.set_yticklabels(labels, fontsize=9)
ax3.set_title('Feature Correlation Matrix', fontweight='bold')
for i in range(4):
    for j in range(4):
        ax3.text(j, i, f'{corr.values[i,j]:.2f}',
                 ha='center', va='center', fontsize=10,
                 fontweight='bold',
                 color='black' if abs(corr.values[i,j])<0.7 else 'white')
plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
plt.tight_layout()

os.makedirs('/content/data/plots', exist_ok=True)
PLOT_PATH = '/content/data/plots/eda_overview.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ Plot saved: {PLOT_PATH}')

# Correlation table
print('\nCorrelation matrix:')
print(corr.round(3).to_string())

## 8. Push to GitHub

Commit the final dataset and this notebook to the public repository.

**Prerequisites:**
- A GitHub Personal Access Token (PAT) with `repo` scope
- Create one at: https://github.com/settings/tokens
- Replace `YOUR_GITHUB_PAT_HERE` below before running

> **Security note:** Never commit your PAT to the notebook. Set it as a variable only — do not save the value to disk or outputs.

In [ ]:
import requests, base64, os, json

# ── Credentials (fill in before running) ──────────────
GITHUB_TOKEN    = 'YOUR_GITHUB_PAT_HERE'  # never commit this value
GITHUB_USERNAME = 'vansh-kumar-007'
REPO_NAME       = 'Internet-Rabbit-Hole-Dataset'
# ──────────────────────────────────────────────────────

HEADERS  = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept'       : 'application/vnd.github.v3+json',
}
REPO_URL = f'https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}'

def scrub_and_encode(path):
    """Read a file, scrub any PAT tokens, return base64-encoded bytes."""
    import re
    with open(path, 'rb') as f:
        raw = f.read()
    # Scrub token patterns from notebook JSON
    if path.endswith('.ipynb'):
        text = raw.decode('utf-8')
        for pat in [r'ghp_[A-Za-z0-9]+',
                    r'github_pat_[A-Za-z0-9_]+',
                    r'gho_[A-Za-z0-9]+']:
            text = re.sub(pat, 'YOUR_GITHUB_PAT_HERE', text)
        raw = text.encode('utf-8')
    return base64.b64encode(raw).decode('utf-8')

def push_file(local_path, github_path, message):
    """Push a single file to GitHub, updating if it already exists."""
    encoded  = scrub_and_encode(local_path)
    url      = f'{REPO_URL}/contents/{github_path}'
    existing = requests.get(url, headers=HEADERS)
    payload  = {'message': message, 'content': encoded}
    if existing.status_code == 200:
        payload['sha'] = existing.json()['sha']
    resp = requests.put(url, headers=HEADERS, json=payload)
    if resp.status_code in (200, 201):
        print(f'  ✓ {github_path}')
    else:
        print(f'  ✗ {github_path} — {resp.status_code}: '
              f'{resp.text[:120]}')

print('Pushing files to GitHub...\n')

# Push the clean notebook
push_file(
    '/content/01_data_collection.ipynb',
    'notebooks/01_data_collection.ipynb',
    'Add clean 01_data_collection notebook (full pipeline)'
)

# Push the EDA plot
push_file(
    '/content/data/plots/eda_overview.png',
    'assets/eda_overview.png',
    'Update EDA overview visualization'
)

print(f'\n✓ Done.')
print(f'  https://github.com/{GITHUB_USERNAME}/{REPO_NAME}')

## Summary

| Step | Output |
|------|--------|
| Download | 484 MB Wikimedia clickstream (May 2026) |
| Filter | 22.2M link-type edges extracted |
| Graph | 3.3M articles, 221 MB RAM (CSR format) |
| Walks | 50,000 sessions, mean 13.3 hops, 78.6% success |
| Features | 3 TF-IDF drift scores per session |
| Dataset | 50,000 rows × 9 columns, 14.5 MB Parquet |

**Kaggle dataset:** https://www.kaggle.com/datasets/vanshkumar007/internet-rabbit-hole-dataset  
**GitHub:** https://github.com/vansh-kumar-007/Internet-Rabbit-Hole-Dataset

### Citation
```
Wikimedia Foundation. (2026). Wikipedia Clickstream.
https://dumps.wikimedia.org/other/clickstream/
Licensed under CC BY-SA 4.0
```